# Carpet diagram — lgm temperature benchmarking

Step toward an update of IPCC AR5 WG1 Fig. 9.12 (a *portrait plot* / *carpet
diagram*). For each PMIP lgm (Last Glacial Maximum (21 ka)) simulation we:

1. Load the annual-mean near-surface temperature field `tas_spatialmean_ann` from the
   CVDP output, and the matching `piControl` run.
2. Compute the **lgm − piControl anomaly** on each model's native grid.
3. **Sample** that anomaly field at the location of every proxy reconstruction.
4. Summarise the model–data mismatch as the **root-mean-squared error (RMSE)** against
   each reconstruction compilation (Cleator (Cleator et al. 2020) and Osman (LGMR; Osman et al. 2021)).

Intermediate tables are written to `output/`. Run with the `my-cli-py` conda env.

In [1]:
import os, glob, re
import numpy as np
import pandas as pd
import xarray as xr

ROOT      = os.getcwd()  # run the notebook from the carpet_diagram/ directory
CVDP_DIR  = os.path.join(ROOT, 'cvdp_output_by_experiment')
EXPERIMENT = 'lgm'
RECON_DIR = os.path.join(ROOT, 'recons', EXPERIMENT)
OUT_DIR   = os.path.join(ROOT, 'output')
os.makedirs(OUT_DIR, exist_ok=True)

VAR        = 'tas_spatialmean_ann'
print('experiment:', EXPERIMENT)
print('CVDP   :', CVDP_DIR)
print('recons :', RECON_DIR)
print('output :', OUT_DIR)

experiment: lgm
CVDP   : /home/ucfaccb@ad.ucl.ac.uk/Documents/local_repos/PMIP7-vision/carpet_diagram/cvdp_output_by_experiment
recons : /home/ucfaccb@ad.ucl.ac.uk/Documents/local_repos/PMIP7-vision/carpet_diagram/recons/lgm
output : /home/ucfaccb@ad.ucl.ac.uk/Documents/local_repos/PMIP7-vision/carpet_diagram/output


## Step 1 — Reconstruction compilations

Two LGM (21 ka) compilations, both near-global gridded anomaly products (so each point
gets a `cos(latitude)` weight for an area-fair RMSE):

- **Cleator** — the Cleator et al. (2020) vegetation-model-inversion data assimilation.
  A long metadata preamble precedes a `#`-prefixed header row (line 77); the only relevant
  field is `MAT` (mean-annual-temperature anomaly, °C) at each land grid cell.
- **Osman** — the Last Glacial Maximum Reanalysis (Osman et al. 2021), the ensemble-mean
  `sat` field from `Osman_LGMR_21ka_SAT_anom_climo.nc`, already differenced to a 21 ka −
  (0–1 ka) anomaly. The 96×144 grid is flattened to points, dropping fill values.

In [2]:
# --- Cleator: gridded land MAT anomalies from a CSV with a preamble ---
CLE_COLS = ['lat', 'lon', 'MI', 'MAP', 'MAT', 'MTCO', 'MTWA', 'GDD5',
            'MI_SD', 'MAP_SD', 'MAT_SD', 'MTCO_SD', 'MTWA_SD', 'GDD5_SD']
cle = pd.read_csv(os.path.join(RECON_DIR, 'cleator2020_recon.csv'),
                  skiprows=77, header=None, names=CLE_COLS)
cle = cle.apply(pd.to_numeric, errors='coerce').dropna(subset=['lat', 'lon', 'MAT'])
cleator = pd.DataFrame({
    'compilation': 'Cleator', 'reference': 'Cleator et al. 2020', 'site': np.nan,
    'Proxy': 'pollen MAT (assim.)',
    'Latitude': cle['lat'].astype(float), 'Longitude': cle['lon'].astype(float),
    'Anom': cle['MAT'].astype(float),
})
cleator['source_table'] = 'cleator2020_recon.csv'

# --- Osman: flatten the gridded LGMR SAT anomaly field to points ---
ds = xr.open_dataset(os.path.join(RECON_DIR, 'Osman_LGMR_21ka_SAT_anom_climo.nc'),
                     decode_times=False)
sat = ds['sat'].where(np.abs(ds['sat']) < 1e30)
lon2d, lat2d = np.meshgrid(sat['lon'].values, sat['lat'].values)
osman = pd.DataFrame({
    'compilation': 'Osman', 'reference': 'Osman et al. 2021', 'site': np.nan,
    'Proxy': 'LGMR SAT (assim.)',
    'Latitude': lat2d.ravel().astype(float), 'Longitude': lon2d.ravel().astype(float),
    'Anom': sat.values.ravel().astype(float),
})
osman['source_table'] = 'Osman_LGMR_21ka_SAT_anom_climo.nc'

recon = pd.concat([cleator, osman], ignore_index=True)
recon = recon.dropna(subset=['Latitude', 'Longitude', 'Anom'])
# Both are regular lat/lon grids: cos-latitude weight makes the RMSE area-fair.
recon['weight'] = np.cos(np.deg2rad(recon['Latitude']))

print(recon.groupby('compilation').size())
recon_out = os.path.join(OUT_DIR, f'recon_points_{EXPERIMENT}.csv')
recon.to_csv(recon_out, index=False)
print('wrote', recon_out)
recon.head()

compilation
Cleator     2214
Osman      13824
dtype: int64
wrote /home/ucfaccb@ad.ucl.ac.uk/Documents/local_repos/PMIP7-vision/carpet_diagram/output/recon_points_lgm.csv


,compilation,reference,site,Proxy,Latitude,Longitude,Anom,source_table,weight
0,Cleator,Cleator et al. 2020,NaN,pollen MAT (assim.),67.0,-179.0,-8.365533,cleator2020_recon.csv,0.390731
1,Cleator,Cleator et al. 2020,NaN,pollen MAT (assim.),71.0,-179.0,-8.286363,cleator2020_recon.csv,0.325568
2,Cleator,Cleator et al. 2020,NaN,pollen MAT (assim.),67.0,-177.0,-9.008762,cleator2020_recon.csv,0.390731
3,Cleator,Cleator et al. 2020,NaN,pollen MAT (assim.),65.0,-175.0,-8.873616,cleator2020_recon.csv,0.422618
4,Cleator,Cleator et al. 2020,NaN,pollen MAT (assim.),65.0,-173.0,-8.879874,cleator2020_recon.csv,0.422618


## Step 2 — Model lgm − piControl anomalies

Each model writes its CVDP field on its own native grid, so anomalies are computed
per model. We pair every `lgm` file with the same model's `piControl` file
(ignoring the auxiliary `.monsoon.` / `.tas.indices.` files). If the two grids ever
differ, the control is bilinearly regridded onto the lgm grid before differencing.

In [3]:
def model_files(experiment):
    """Map model name -> path for the main CVDP file of an experiment."""
    out = {}
    pat = os.path.join(CVDP_DIR, experiment, f'*_{experiment}.cvdp_data.*.nc')
    for f in sorted(glob.glob(pat)):
        b = os.path.basename(f)
        if '.monsoon.' in b or '.tas.indices.' in b or b.endswith('.tmp'):
            continue
        model = b.split(f'_{experiment}.cvdp_data')[0]
        out[model] = f  # last (sorted) wins; there is one main file per model
    return out

def has_var(path):
    """True if the CVDP file actually carries the tas field (a few don't)."""
    with xr.open_dataset(path, decode_times=False) as ds:
        return VAR in ds.variables

exp_files = model_files(EXPERIMENT)
pi_files  = model_files('piControl')
paired = sorted(set(exp_files) & set(pi_files))
# Some CVDP files lack tas_spatialmean_ann entirely — drop those models.
models = [m for m in paired if has_var(exp_files[m]) and has_var(pi_files[m])]
print(f'{len(models)} models with both {EXPERIMENT} and piControl (and a tas field):')
print(models)
missing_pi = sorted(set(exp_files) - set(pi_files))
if missing_pi:
    print(f'{EXPERIMENT} models with no piControl (skipped):', missing_pi)
no_var = [m for m in paired if m not in models]
if no_var:
    print(f'models dropped — no {VAR} in CVDP file:', no_var)

15 models with both lgm and piControl (and a tas field):
['AWI-ESM-1-1-LR', 'CCSM4', 'CESM2-FV2', 'CESM2-WACCM-FV2', 'CNRM-CM5', 'FGOALS-g2', 'GISS-E2-R', 'INM-CM4-8', 'IPSL-CM5A-LR', 'MIROC-ES2L', 'MIROC-ESM', 'MPI-ESM-P', 'MPI-ESM1-2-LR', 'MRI-CGCM3', 'UofT-CCSM-4']
models dropped — no tas_spatialmean_ann in CVDP file: ['COSMOS-ASO']


In [4]:
def load_field(path):
    da = xr.open_dataset(path, decode_times=False)[VAR].sortby('lat').sortby('lon')
    # A few CVDP grids (e.g. LOVECLIM piControl) carry duplicate lon values,
    # which break interpolation; keep the first occurrence of each coordinate.
    for dim in ('lat', 'lon'):
        _, idx = np.unique(da[dim].values, return_index=True)
        if len(idx) != da.sizes[dim]:
            da = da.isel({dim: np.sort(idx)})
    return da

anomalies = {}
for m in models:
    exp_field = load_field(exp_files[m])
    pi  = load_field(pi_files[m])
    if exp_field.shape != pi.shape or not (np.allclose(exp_field.lat, pi.lat) and np.allclose(exp_field.lon, pi.lon)):
        pi = pi.interp(lat=exp_field.lat, lon=exp_field.lon)
    anomalies[m] = (exp_field - pi).rename('tas_anom')
    print(f'{m:18s} grid {exp_field.shape}  mean anom {float(anomalies[m].mean()):+.2f} C')

AWI-ESM-1-1-LR     grid (96, 192)  mean anom -5.12 C


CCSM4              grid (192, 288)  mean anom -6.79 C
CESM2-FV2          grid (96, 144)  mean anom -8.55 C


CESM2-WACCM-FV2    grid (96, 144)  mean anom -9.12 C
CNRM-CM5           grid (128, 256)  mean anom -3.43 C


FGOALS-g2          grid (60, 128)  mean anom -5.93 C
GISS-E2-R          grid (90, 144)  mean anom -6.92 C


INM-CM4-8          grid (120, 180)  mean anom -4.81 C
IPSL-CM5A-LR       grid (96, 96)  mean anom -5.86 C


MIROC-ES2L         grid (64, 128)  mean anom -5.42 C
MIROC-ESM          grid (64, 128)  mean anom -7.09 C


MPI-ESM-P          grid (96, 192)  mean anom -6.15 C


MPI-ESM1-2-LR      grid (96, 192)  mean anom -5.21 C
MRI-CGCM3          grid (160, 320)  mean anom -6.39 C


UofT-CCSM-4        grid (192, 288)  mean anom -7.19 C


## Step 3 — Sample model anomalies at reconstruction locations

Model longitudes run 0–360°, the proxy longitudes −180–180°, so targets are wrapped to
0–360 and the field is made cyclic in longitude before bilinear interpolation.

Each recon point carries a `weight` used later in the RMSE. Scattered-site compilations
weight every point equally (1.0); gridded near-global reconstructions (Cleator, Osman)
set `weight = cos(latitude)` so the RMSE is area-fair rather than pole-heavy.

In [5]:
def sample_points(field, lats, lons):
    """Bilinearly sample a (lat, lon) field at scattered points; lon made cyclic."""
    lon_cyc = np.append(field.lon.values, field.lon.values[0] + 360.0)
    fcyc = xr.concat([field, field.isel(lon=0)], dim='lon').assign_coords(lon=lon_cyc)
    ta = xr.DataArray(np.asarray(lats), dims='point')
    to = xr.DataArray(np.asarray(lons) % 360.0, dims='point')
    return fcyc.interp(lat=ta, lon=to).values

sampled = recon[['compilation', 'reference', 'site', 'Proxy',
                 'Latitude', 'Longitude', 'Anom']].copy()
sampled = sampled.rename(columns={'Anom': 'recon_anom'})
# Optional per-point weight (defaults to equal weighting when Step 1 omits it).
sampled['weight'] = recon['weight'].values if 'weight' in recon.columns else 1.0
for m in models:
    sampled[m] = sample_points(anomalies[m], sampled['Latitude'].values, sampled['Longitude'].values)

sampled_out = os.path.join(OUT_DIR, f'model_anom_at_recon_{EXPERIMENT}.csv')
sampled.to_csv(sampled_out, index=False)
print('wrote', sampled_out, '  shape', sampled.shape)
sampled.head()

wrote /home/ucfaccb@ad.ucl.ac.uk/Documents/local_repos/PMIP7-vision/carpet_diagram/output/model_anom_at_recon_lgm.csv   shape (16038, 23)


,compilation,reference,site,Proxy,Latitude,Longitude,recon_anom,weight,AWI-ESM-1-1-LR,CCSM4,...,FGOALS-g2,GISS-E2-R,INM-CM4-8,IPSL-CM5A-LR,MIROC-ES2L,MIROC-ESM,MPI-ESM-P,MPI-ESM1-2-LR,MRI-CGCM3,UofT-CCSM-4
0,Cleator,Cleator et al. 2020,NaN,pollen MAT (assim.),67.0,-179.0,-8.365533,0.390731,-3.314389,-8.467511,...,-11.422608,-16.364082,-2.422333,-4.544382,-6.680903,-10.047369,-4.742196,-2.991713,-6.448630,-8.844007
1,Cleator,Cleator et al. 2020,NaN,pollen MAT (assim.),71.0,-179.0,-8.286363,0.325568,-3.976460,-10.945594,...,-13.382867,-18.657314,-0.297595,-3.586125,-6.527646,-10.498195,-5.232141,-3.180337,-8.000159,-12.536451
2,Cleator,Cleator et al. 2020,NaN,pollen MAT (assim.),67.0,-177.0,-9.008762,0.390731,-3.357998,-9.355254,...,-12.814205,-17.218596,-2.347160,-4.440582,-6.597902,-10.199751,-4.780364,-2.857989,-7.394926,-11.079180
3,Cleator,Cleator et al. 2020,NaN,pollen MAT (assim.),65.0,-175.0,-8.873616,0.422618,-4.598341,-9.267862,...,-10.241200,-15.053536,-2.625724,-4.798518,-6.300106,-10.044648,-4.619277,-2.829328,-6.878565,-11.329933
4,Cleator,Cleator et al. 2020,NaN,pollen MAT (assim.),65.0,-173.0,-8.879874,0.422618,-4.235348,-8.888150,...,-10.418143,-14.827534,-2.095846,-4.612541,-6.034313,-10.087180,-4.493904,-2.510354,-6.752924,-11.547300


## Step 4 — RMSE of each model against each compilation

For every model × compilation we take the model-minus-proxy difference across all proxy
points in that compilation and report the (weight-weighted) RMSE and bias, plus the number
of points contributing (a point off the model grid edge can return NaN). With unit weights
this is the ordinary RMSE; gridded compilations use `cos(latitude)` weights (Step 3).

In [6]:
records = []
for m in models:
    for comp, grp in sampled.groupby('compilation'):
        diff = grp[m].values - grp['recon_anom'].values
        w = grp['weight'].values
        valid = np.isfinite(diff) & np.isfinite(w)
        n = int(valid.sum())
        if n:
            dv, wv = diff[valid], w[valid]
            rmse = float(np.sqrt(np.sum(wv * dv ** 2) / np.sum(wv)))
            bias = float(np.sum(wv * dv) / np.sum(wv))
        else:
            rmse = bias = np.nan
        records.append({'model': m, 'compilation': comp, 'n_points': n,
                        'rmse': rmse, 'bias': bias})

rmse_long = pd.DataFrame(records)
rmse_wide = rmse_long.pivot(index='model', columns='compilation', values='rmse')
rmse_wide.columns = [f'{c}_RMSE' for c in rmse_wide.columns]

rmse_long.to_csv(os.path.join(OUT_DIR, f'rmse_long_{EXPERIMENT}.csv'), index=False)
rmse_wide.to_csv(os.path.join(OUT_DIR, f'rmse_summary_{EXPERIMENT}.csv'))
print(f'wrote rmse_long_{EXPERIMENT}.csv and rmse_summary_{EXPERIMENT}.csv')
rmse_wide.sort_values(rmse_wide.columns[0])

wrote rmse_long_lgm.csv and rmse_summary_lgm.csv


,Cleator_RMSE,Osman_RMSE
model,,
IPSL-CM5A-LR,2.293258,4.423680
INM-CM4-8,2.386247,5.005875
MPI-ESM-P,2.680045,4.104728
MIROC-ES2L,2.691053,4.361049
CCSM4,2.780847,3.663350
FGOALS-g2,2.796614,3.961142
MPI-ESM1-2-LR,2.970425,4.745336
MRI-CGCM3,2.974014,4.437757
MIROC-ESM,3.105006,3.359862


These RMSE values are the building blocks of the carpet diagram: one column per model,
one row per (period, reconstruction compilation), coloured by RMSE. `carpet_figure.py`
reads every `output/rmse_long_<period>.csv`, so re-running this notebook for a new period
makes its rows appear in the portrait plot automatically.